# EscutIA — mesclar e subir o QLoRA

Este notebook recebe o arquivo qlora.zip, mescla o adapter ao modelo-base em uma GPU do Google Colab e publica o **modelo completo** no Hugging Face Hub. O resultado pode ser carregado diretamente com transformers, sem PEFT.

> Antes de começar, selecione **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**. Não escreva o token no código: prefira o segredo HF_TOKEN do Colab; se ele não existir, o notebook abrirá um campo oculto.

In [ ]:
# Gradio e torchao são opcionais e suas versões pré-instaladas podem conflitar.
# Nenhum deles é necessário para mesclar ou publicar este QLoRA.
!pip -q uninstall -y gradio gradio_client torchao
!pip -q install -U "transformers>=4.46,<5" "peft>=0.18,<1" "accelerate>=1.0,<2" "huggingface_hub>=0.27,<1" safetensors

## 1. Autenticar e configurar a publicação

Informe apenas o nome do repositório em NOME_REPOSITORIO. O notebook usa automaticamente o usuário pertencente ao token e publica como público por padrão. Altere PRIVADO para True somente se quiser um repositório privado.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi

NOME_REPOSITORIO = "escutia-qlora-completo"
PRIVADO = False

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass("Token do Hugging Face (permissão de escrita): ").strip()
if not HF_TOKEN:
    raise ValueError("Token não informado.")

api = HfApi(token=HF_TOKEN)
identidade = api.whoami()
usuario = identidade.get("name") or identidade.get("fullname")
if not usuario:
    raise RuntimeError("Não foi possível identificar o proprietário do token.")
REPO_ID = f"{usuario}/{NOME_REPOSITORIO}"
print(f"Autenticado como: {usuario}")
print(f"Destino: {REPO_ID} ({'privado' if PRIVADO else 'público'})")

## 2. Enviar e validar o pacote QLoRA

Se qlora.zip já estiver em /content, ele será reutilizado automaticamente. Caso contrário, o seletor de upload será aberto. O ZIP pode conter os arquivos na raiz ou dentro de uma única pasta, como escutia-qlora/.

In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path
from google.colab import files

AREA_TRABALHO = Path("/content/escutia_qlora_merge")
ADAPTER_DIR = AREA_TRABALHO / "adapter"
SAIDA_DIR = AREA_TRABALHO / "modelo_completo"

if AREA_TRABALHO.exists():
    shutil.rmtree(AREA_TRABALHO)
ADAPTER_DIR.mkdir(parents=True)

zip_path = Path("/content/qlora.zip")
if zip_path.is_file():
    print(f"Reutilizando ZIP já enviado: {zip_path}")
else:
    print("qlora.zip não encontrado em /content. Selecione o arquivo para enviar.")
    enviados = files.upload()
    arquivos_zip = [Path(nome) for nome in enviados if nome.lower().endswith(".zip")]
    if len(arquivos_zip) != 1:
        raise ValueError("Envie exatamente um arquivo ZIP: qlora.zip.")
    arquivo_enviado = arquivos_zip[0]
    if arquivo_enviado.resolve() != zip_path.resolve():
        shutil.move(str(arquivo_enviado), str(zip_path))
    print(f"ZIP armazenado para reutilização nesta sessão: {zip_path}")

with zipfile.ZipFile(zip_path) as pacote:
    for membro in pacote.infolist():
        destino = (ADAPTER_DIR / membro.filename).resolve()
        if ADAPTER_DIR.resolve() not in destino.parents and destino != ADAPTER_DIR.resolve():
            raise ValueError(f"Caminho inseguro no ZIP: {membro.filename}")
    pacote.extractall(ADAPTER_DIR)

configs = list(ADAPTER_DIR.rglob("adapter_config.json"))
if len(configs) != 1:
    raise FileNotFoundError("O ZIP deve conter exatamente um adapter_config.json.")
ADAPTER_DIR = configs[0].parent
pesos = list(ADAPTER_DIR.glob("adapter_model.*"))
if not pesos:
    raise FileNotFoundError("Pesos adapter_model não encontrados no ZIP.")

adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
MODELO_BASE = adapter_config.get("base_model_name_or_path")
REVISAO_BASE = adapter_config.get("revision") or None
if not REVISAO_BASE:
    arquivos_yaml = list(ADAPTER_DIR.glob("*.yaml")) + list(ADAPTER_DIR.glob("*.yml"))
    for arquivo_yaml in arquivos_yaml:
        correspondencia = re.search(
            r"(?m)^\s*model_revision\s*:\s*['\"]?([^\s'\"#]+)",
            arquivo_yaml.read_text(encoding="utf-8"),
        )
        if correspondencia:
            REVISAO_BASE = correspondencia.group(1)
            break
if not MODELO_BASE:
    raise ValueError("O adapter_config.json não informa base_model_name_or_path.")

print(f"Adapter: {ADAPTER_DIR}")
print(f"Modelo-base detectado: {MODELO_BASE}")
print(f"Revisão: {REVISAO_BASE or 'padrão do repositório'}")

## 3. Mesclar o adapter

O modelo-base é carregado em float16, e não em 4 bits. A quantização economizou memória durante o treinamento; o merge incorpora as matrizes aprendidas aos pesos reais do modelo.

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Libera objetos de uma tentativa anterior antes de recarregar os pesos.
for nome_objeto in ("base", "modelo_peft", "modelo_completo"):
    if nome_objeto in globals():
        del globals()[nome_objeto]
gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("GPU não detectada. Ative uma GPU no ambiente de execução do Colab.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    revision=REVISAO_BASE,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
modelo_peft = PeftModel.from_pretrained(base, str(ADAPTER_DIR), token=HF_TOKEN)
modelo_completo = modelo_peft.merge_and_unload(safe_merge=True)

# O adapter reutiliza o vocabulário do Qwen. Carregar do modelo-base evita
# incompatibilidades em tokenizer_config.json gerados por outras versões.
tokenizer = AutoTokenizer.from_pretrained(
    MODELO_BASE,
    revision=REVISAO_BASE,
    token=HF_TOKEN,
)

SAIDA_DIR.mkdir(parents=True, exist_ok=False)
modelo_completo.save_pretrained(SAIDA_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(SAIDA_DIR)
print(f"Modelo completo salvo em: {SAIDA_DIR}")

## 4. Criar o Model Card e publicar

A célula cria o repositório se necessário e envia a pasta completa. Se o repositório já existir, os arquivos de mesmo nome serão atualizados.

In [ ]:
model_card = f'''---
base_model: {MODELO_BASE}
base_model_relation: finetune
library_name: transformers
language:
- pt
pipeline_tag: text-generation
tags:
- qlora
- merged
- sentiment-analysis
- portuguese
---

# EscutIA QLoRA — modelo completo

Modelo para classificação de sentimentos em português, treinado com QLoRA e mesclado ao modelo-base {MODELO_BASE}.

Este repositório contém os pesos completos e pode ser carregado diretamente com transformers; não é necessário aplicar um adapter PEFT separado.

Exemplo:

    from transformers import AutoModelForCausalLM, AutoTokenizer
    model_id = "{REPO_ID}"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
'''
(SAIDA_DIR / "README.md").write_text(model_card, encoding="utf-8")

api.create_repo(repo_id=REPO_ID, repo_type="model", private=PRIVADO, exist_ok=True)
api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(SAIDA_DIR),
    commit_message="Publica modelo QLoRA mesclado",
)
print(f"Publicação concluída: https://huggingface.co/{REPO_ID}")

## 5. Verificação rápida (opcional)

Confirma que o Hub possui a configuração necessária sem carregar uma segunda cópia do modelo na GPU.

In [ ]:
from transformers import AutoConfig

config_publicada = AutoConfig.from_pretrained(REPO_ID, token=HF_TOKEN)
print(f"Modelo publicado e acessível: {REPO_ID}")
print(f"Arquitetura: {config_publicada.architectures}")